# Phase 1: Exploratory Data Analysis

## Goals
- Load and inspect both fraud detection datasets
- Understand class imbalance, feature distributions, and temporal patterns
- Create temporal train/cal/val/test splits
- Establish baseline understanding before modeling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.helpers import load_config, set_seed

config = load_config('../configs/config.yaml')
set_seed(config['experiment']['seed'])

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Setup complete.')

## 1. Load Credit Card Fraud Dataset

In [ ]:
# Load Credit Card Fraud dataset
cc_path = f"../{config['data']['creditcard_path']}/creditcard.csv"
df_cc = pd.read_csv(cc_path)
print(f'Shape: {df_cc.shape}')
print(f'\nClass distribution:')
print(df_cc['Class'].value_counts())
print(f'\nFraud rate: {df_cc["Class"].mean():.4f} ({df_cc["Class"].mean()*100:.2f}%)')
df_cc.head()

In [ ]:
# Basic statistics
df_cc.describe()

In [ ]:
# Missing values
print('Missing values per column:')
missing = df_cc.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')

## 2. Class Imbalance Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
counts = df_cc['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#4CAF50', '#F44336'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontweight='bold')

# Log scale
axes[1].bar(['Legitimate', 'Fraud'], counts.values, color=['#4CAF50', '#F44336'])
axes[1].set_yscale('log')
axes[1].set_title('Class Distribution (Log Scale)')
axes[1].set_ylabel('Count (log)')

plt.suptitle(f'Class Imbalance — Fraud Rate: {df_cc["Class"].mean()*100:.3f}%', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/class_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Distributions by Class

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, cls_name, cls_val in [(0, 'Legitimate', 0), (1, 'Fraud', 1)]:
    subset = df_cc[df_cc['Class'] == cls_val]
    axes[0].hist(subset['Amount'], bins=50, alpha=0.6, label=cls_name, density=True)
    axes[1].hist(np.log1p(subset['Amount']), bins=50, alpha=0.6, label=cls_name, density=True)

axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

axes[1].set_title('Log(Amount) Distribution')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA feature distributions (V1-V28) — show most discriminative
fig, axes = plt.subplots(3, 4, figsize=(16, 10))

# Find features with highest KS statistic between classes
from scipy.stats import ks_2samp
v_cols = [f'V{i}' for i in range(1, 29)]
ks_stats = {}
for col in v_cols:
    stat, _ = ks_2samp(
        df_cc[df_cc['Class']==0][col].dropna(),
        df_cc[df_cc['Class']==1][col].dropna()
    )
    ks_stats[col] = stat

top_features = sorted(ks_stats, key=ks_stats.get, reverse=True)[:12]

for ax, feat in zip(axes.flat, top_features):
    for cls, label, color in [(0, 'Legit', '#4CAF50'), (1, 'Fraud', '#F44336')]:
        ax.hist(df_cc[df_cc['Class']==cls][feat], bins=50, alpha=0.5,
                label=label, density=True, color=color)
    ax.set_title(f'{feat} (KS={ks_stats[feat]:.3f})')
    ax.legend(fontsize=8)

plt.suptitle('Top Discriminative Features (by KS statistic)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Temporal Patterns

In [ ]:
# Time-based fraud rate analysis
df_cc['Time_hours'] = df_cc['Time'] / 3600

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Transaction volume over time
axes[0].hist(df_cc[df_cc['Class']==0]['Time_hours'], bins=100, alpha=0.6, label='Legit')
axes[0].hist(df_cc[df_cc['Class']==1]['Time_hours'], bins=100, alpha=0.6, label='Fraud')
axes[0].set_title('Transaction Volume Over Time')
axes[0].set_xlabel('Time (hours)')
axes[0].legend()

# Rolling fraud rate
df_sorted = df_cc.sort_values('Time')
window = 5000
rolling_fraud_rate = df_sorted['Class'].rolling(window).mean()
axes[1].plot(df_sorted['Time_hours'].values, rolling_fraud_rate.values, color='red', alpha=0.7)
axes[1].set_title(f'Rolling Fraud Rate (window={window})')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Fraud Rate')
axes[1].axhline(y=df_cc['Class'].mean(), color='black', linestyle='--', label='Overall rate')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Create Temporal Splits

In [ ]:
from src.data.splits import temporal_split_creditcard

splits = temporal_split_creditcard(df_cc)

print('Temporal Split Summary:')
print('=' * 50)
for name, idx in splits.items():
    y_split = df_cc.iloc[idx]['Class'].values
    time_range = df_cc.iloc[idx]['Time_hours']
    print(f'{name:15s}: n={len(idx):6d}, fraud_rate={y_split.mean():.4f}, '
          f'time=[{time_range.min():.1f}h - {time_range.max():.1f}h]')

In [ ]:
# Visualize temporal splits
fig, ax = plt.subplots(figsize=(14, 4))

colors = {'train': '#2196F3', 'calibration': '#FF9800', 'val': '#9C27B0', 'test': '#F44336'}

for name, idx in splits.items():
    times = df_cc.iloc[idx]['Time_hours'].values
    ax.axvspan(times.min(), times.max(), alpha=0.3, color=colors[name], label=name)

ax.set_xlabel('Time (hours)')
ax.set_title('Temporal Split Boundaries')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/temporal_splits.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Correlations

In [ ]:
# Correlation with target
correlations = df_cc.drop(columns=['Time_hours']).corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#F44336' if v < 0 else '#4CAF50' for v in correlations.values]
ax.barh(correlations.index, correlations.values, color=colors)
ax.set_xlabel('Correlation with Fraud Label')
ax.set_title('Feature Correlation with Target')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../results/figures/feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Observations for Research

Document your findings here:

1. **Class Imbalance**: Fraud rate is ~0.17%, creating extreme imbalance
2. **Temporal Patterns**: Does fraud rate change over time? (key for shift experiments)
3. **Feature Separability**: Which features best distinguish fraud from legit?
4. **Implications for Calibration**: High imbalance means most bins in reliability diagram will be near 0